In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from pathlib import Path

RAW      = Path('../data/01_raw')
PROCESSED = Path('../data/02_processed')
FEATURES  = Path('../data/03_features')
SLUG      = 'esm2_t33_650M_UR50D'

pairs       = pd.read_csv(PROCESSED / 'train_mutant_pairs.csv')
test_labels = pd.read_csv(RAW / 'test_labels.csv')
test        = pd.read_csv(RAW / 'test.csv').merge(test_labels, on='seq_id')

print(f'train pairs: {len(pairs)}, test: {len(test)}')

In [ ]:
# Delta Tm: deviation from cluster consensus Tm (grouped by derived wildtype sequence)
# For test, all mutations share one wildtype — use median test Tm as proxy
cluster_mean = pairs.groupby('wildtype_sequence')['tm'].transform('mean')
delta_train  = pairs['tm'] - cluster_mean
delta_test   = test['tm'] - test['tm'].median()

fig, ax = plt.subplots(figsize=(9, 4))
bins = np.linspace(-40, 40, 60)
ax.hist(delta_train, bins=bins, alpha=0.6, color='steelblue', density=True, label=f'train pairs (n={len(delta_train)})')
ax.hist(delta_test,  bins=bins, alpha=0.6, color='tomato',    density=True, label=f'test (n={len(delta_test)})')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('ΔTm (°C) relative to cluster/wildtype')
ax.set_ylabel('density')
ax.set_title('ΔTm distribution: train pairs vs test')
ax.legend()
plt.tight_layout()
plt.show()

print(f'train  ΔTm  mean={delta_train.mean():.2f}  std={delta_train.std():.2f}  range=[{delta_train.min():.1f}, {delta_train.max():.1f}]')
print(f'test   ΔTm  mean={delta_test.mean():.2f}  std={delta_test.std():.2f}  range=[{delta_test.min():.1f}, {delta_test.max():.1f}]')

In [ ]:
# t-SNE on delta embeddings (mut - wt) for train pairs vs test
# Subsample train to keep runtime reasonable
X_train_mut = np.load(FEATURES / f'{SLUG}_train_mutant.npy')
X_train_wt  = np.load(FEATURES / f'{SLUG}_train_wildtype.npy')
X_val_mut   = np.load(FEATURES / f'{SLUG}_val_mutant.npy')
X_val_wt    = np.load(FEATURES / f'{SLUG}_val_wildtype.npy')
X_test_mut  = np.load(FEATURES / f'{SLUG}_test_mutant.npy')
X_test_wt   = np.load(FEATURES / f'{SLUG}_test_wildtype.npy')

delta_train_emb = np.vstack([X_train_mut - X_train_wt, X_val_mut - X_val_wt])
delta_test_emb  = X_test_mut - X_test_wt

rng = np.random.default_rng(42)
idx = rng.choice(len(delta_train_emb), size=min(2000, len(delta_train_emb)), replace=False)
X_combined = np.vstack([delta_train_emb[idx], delta_test_emb])
labels = np.array(['train'] * len(idx) + ['test'] * len(delta_test_emb))

print(f'Running t-SNE on {len(X_combined)} points...')
Z = TSNE(n_components=2, random_state=42, perplexity=40, n_iter=1000).fit_transform(X_combined)

fig, ax = plt.subplots(figsize=(8, 6))
for group, color in [('train', 'steelblue'), ('test', 'tomato')]:
    mask = labels == group
    ax.scatter(Z[mask, 0], Z[mask, 1], c=color, label=group, alpha=0.4, s=8, linewidths=0)
ax.legend(markerscale=2)
ax.set_title('t-SNE of delta embeddings (mut − wt): train pairs vs test')
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()